# UC1 — Feature Engineering: Asistente Financiero Proactivo

**Autor:** Fernando Haro (`fh`)  
**Proyecto:** datamoles — Havi, copiloto financiero proactivo para Hey Banco  
**UC:** UC1 — Detección de anomalías y alertas proactivas al usuario  

## Objetivo

Construir la tabla de features por usuario (`feat_uc1_user_risk.parquet`) y la tabla de alertas por transacción (`feat_uc1_alertas.parquet`) que alimentarán el modelo de anomalías de UC1.

Las features se agrupan en 4 bloques:
1. **Rechazo** — patrones de transacciones no procesadas (saldo insuficiente, límite excedido)
2. **Atipicidad** — comportamiento inusual a nivel transacción y usuario
3. **Liquidez** — capacidad del usuario para cubrir compromisos
4. **Conversación** — señales UC4 que correlacionan con problemas financieros

**Fecha de corte:** `2025-10-31` — todo lo posterior se descarta para evitar data leakage.

---
## 0. Setup

In [1]:
# Author: Fernando Haro | UC1 — Feature Engineering
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# --- Parámetros globales ---
# Fecha de corte: todo lo posterior se descarta (evita data leakage)
CUTOFF_DATE = pd.Timestamp("2025-10-31")

# Ventanas temporales para features de rechazo
WINDOW_30D = 30
WINDOW_90D = 90

# Rutas absolutas a los datasets
BASE_TXN  = Path("/Users/diegodq/Documents/dev/datamoles/Datathon-2026/Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones")
BASE_CONV = Path("/Users/diegodq/Documents/dev/datamoles/Datathon-2026/Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones")
OUTPUT_DIR = Path("/Users/diegodq/Documents/dev/datamoles/Datathon-2026/outputs/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"CUTOFF_DATE : {CUTOFF_DATE.date()}")
print(f"BASE_TXN    : {BASE_TXN}")
print(f"BASE_CONV   : {BASE_CONV}")
print(f"OUTPUT_DIR  : {OUTPUT_DIR}")

CUTOFF_DATE : 2025-10-31
BASE_TXN    : /Users/diegodq/Documents/dev/datamoles/Datathon-2026/Datathon_Hey_2026_dataset_transacciones 1/dataset_transacciones
BASE_CONV   : /Users/diegodq/Documents/dev/datamoles/Datathon-2026/Datathon_Hey_dataset_conversaciones 1/dataset_conversaciones
OUTPUT_DIR  : /Users/diegodq/Documents/dev/datamoles/Datathon-2026/outputs/features


---
## 1. Carga de datos

Cargamos los tres CSVs principales. Aplicamos las correcciones conocidas del dataset:
- Eliminamos `es_dato_sintetico` (columna de flag sintético, no aporta señal real)
- Renombramos `patron_uso_atipico` en clientes y transacciones para evitar colisión en joins
- Parseamos `fecha_hora` como datetime y filtramos al CUTOFF_DATE

In [2]:
%%time
# --- Clientes ---
df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv")

# Eliminar columna de flag sintético si existe
if 'es_dato_sintetico' in df_clientes.columns:
    df_clientes.drop(columns=['es_dato_sintetico'], inplace=True)

# Renombrar patron_uso_atipico para evitar colisión con la misma columna en transacciones
if 'patron_uso_atipico' in df_clientes.columns:
    df_clientes.rename(columns={'patron_uso_atipico': 'patron_uso_atipico_user'}, inplace=True)

print(f"df_clientes  : {df_clientes.shape}")
df_clientes.head(2)

df_clientes  : (15025, 22)
CPU times: user 22.1 ms, sys: 4.09 ms, total: 26.2 ms
Wall time: 27.4 ms


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,...,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico_user
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,...,527,1,app_android,10.0,False,True,es_MX,False,2,False
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,...,714,3,app_android,8.0,False,True,es_MX,True,2,False


In [3]:
%%time
# --- Productos ---
df_productos = pd.read_csv(BASE_TXN / "hey_productos.csv")

if 'es_dato_sintetico' in df_productos.columns:
    df_productos.drop(columns=['es_dato_sintetico'], inplace=True)

print(f"df_productos : {df_productos.shape}")
df_productos.head(2)

df_productos : (38909, 12)
CPU times: user 28.4 ms, sys: 5.32 ms, total: 33.7 ms
Wall time: 34.5 ms


,producto_id,user_id,tipo_producto,fecha_apertura,estatus,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento
0,PRD-00000001,USR-00001,cuenta_debito,2023-06-26,activo,NaN,80954.6,NaN,NaN,NaN,NaN,2025-11-27
1,PRD-00000002,USR-00001,tarjeta_credito_hey,2022-10-16,activo,144000.0,88790.4,0.6166,35.71,NaN,NaN,2025-09-17


In [4]:
%%time
# --- Transacciones ---
df_tx = pd.read_csv(BASE_TXN / "hey_transacciones.csv")

if 'es_dato_sintetico' in df_tx.columns:
    df_tx.drop(columns=['es_dato_sintetico'], inplace=True)

# La misma columna existe en transacciones con semántica distinta (nivel transacción, no usuario)
if 'patron_uso_atipico' in df_tx.columns:
    df_tx.rename(columns={'patron_uso_atipico': 'patron_uso_atipico_txn'}, inplace=True)

# Parsear fecha como datetime para poder hacer aritmética temporal
df_tx['fecha_hora'] = pd.to_datetime(df_tx['fecha_hora'])

# Filtrar al CUTOFF_DATE — todo lo posterior podría filtrar hacia el futuro
n_antes = len(df_tx)
df_tx = df_tx[df_tx['fecha_hora'] <= CUTOFF_DATE]
n_despues = len(df_tx)

print(f"df_tx antes del filtro  : {n_antes:,} filas")
print(f"df_tx después del filtro: {n_despues:,} filas  (descartadas: {n_antes - n_despues:,})")
print(f"Rango de fechas: {df_tx['fecha_hora'].min().date()} → {df_tx['fecha_hora'].max().date()}")
df_tx.head(2)

df_tx antes del filtro  : 802,384 filas
df_tx después del filtro: 731,033 filas  (descartadas: 71,351)
Rango de fechas: 2025-01-01 → 2025-10-30
CPU times: user 1.55 s, sys: 137 ms, total: 1.69 s
Wall time: 1.7 s


,transaccion_id,user_id,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,...,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico_txn
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.88,DivertidoPark,entretenimiento,"Nueva York, NY",...,NaN,1,NaN,0.34,Cargo automático,14,Wednesday,True,app_ios,False
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.00,GamerPass,servicios_digitales,CDMX - Benito Juárez,...,NaN,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False


---
## 2. Features de rechazo (ventanas 30d y 90d)

Las transacciones rechazadas son la señal más directa de estrés financiero. Distinguimos dos niveles:
- **Rechazo general** (`no_procesada`): cualquier rechazo
- **Rechazo UC1-accionable** (`saldo_insuficiente` + `limite_excedido`): Havi puede actuar sobre estos

Usamos ventanas de 30 y 90 días para capturar tanto urgencia (30d) como cronicidad (90d).

In [5]:
# Fechas de inicio para cada ventana temporal
start_30d = CUTOFF_DATE - pd.Timedelta(days=WINDOW_30D)
start_90d = CUTOFF_DATE - pd.Timedelta(days=WINDOW_90D)

# Motivos de rechazo sobre los cuales Havi puede actuar (ej: transferir desde inversión)
MOTIVOS_UC1 = ['saldo_insuficiente', 'limite_excedido']

# Subconjuntos temporales
df_30d = df_tx[df_tx['fecha_hora'] >= start_30d].copy()
df_90d = df_tx[df_tx['fecha_hora'] >= start_90d].copy()

print(f"Transacciones últimos 30d: {len(df_30d):,}")
print(f"Transacciones últimos 90d: {len(df_90d):,}")

Transacciones últimos 30d: 73,548
Transacciones últimos 90d: 218,932


In [6]:
# --- Rechazos en ventana 30d ---
rechazos_30d = df_30d[df_30d['estatus'] == 'no_procesada'].groupby('user_id')

feat_rechazos_30d = pd.DataFrame({
    # Total de rechazos en 30d (cualquier motivo)
    'feat_n_rechazos_30d': rechazos_30d.size(),
    # Solo rechazos accionables por UC1
    'feat_n_rechazos_uc1_30d': df_30d[
        (df_30d['estatus'] == 'no_procesada') &
        (df_30d['motivo_no_procesada'].isin(MOTIVOS_UC1))
    ].groupby('user_id').size(),
}).fillna(0).astype(int).reset_index()

# --- Rechazos en ventana 90d ---
rechazos_90d = df_90d[df_90d['estatus'] == 'no_procesada'].groupby('user_id')

feat_rechazos_90d = pd.DataFrame({
    'feat_n_rechazos_90d': rechazos_90d.size(),
    'feat_n_rechazos_uc1_90d': df_90d[
        (df_90d['estatus'] == 'no_procesada') &
        (df_90d['motivo_no_procesada'].isin(MOTIVOS_UC1))
    ].groupby('user_id').size(),
}).fillna(0).astype(int).reset_index()

print(f"Usuarios con rechazos 30d: {len(feat_rechazos_30d):,}")
print(f"Usuarios con rechazos 90d: {len(feat_rechazos_90d):,}")

Usuarios con rechazos 30d: 2,170
Usuarios con rechazos 90d: 5,384


In [7]:
# --- Features adicionales sobre todo el período hasta CUTOFF ---

# Porcentaje de reintentos: indica usuarios que intentan la misma transacción repetidamente
# Es señal de usuario que sabe que tiene fondos insuficientes pero lo intenta igual
n_reintentos = df_tx[df_tx['intento_numero'] > 1].groupby('user_id').size().rename('_n_reintentos')
n_total_tx   = df_tx.groupby('user_id').size().rename('_n_total')
feat_reintento = pd.concat([n_reintentos, n_total_tx], axis=1).fillna(0)
feat_reintento['feat_pct_reintento'] = (feat_reintento['_n_reintentos'] / feat_reintento['_n_total']).fillna(0)
feat_reintento = feat_reintento[['feat_pct_reintento']].reset_index()

# Días desde el último rechazo hasta CUTOFF (urgencia de la alerta)
df_rechazos_all = df_tx[df_tx['estatus'] == 'no_procesada']
ult_rechazo = df_rechazos_all.groupby('user_id')['fecha_hora'].max().rename('_ult_rechazo_fecha')
feat_dist_rechazo = ult_rechazo.to_frame()
feat_dist_rechazo['feat_dist_dias_ult_rechazo'] = (CUTOFF_DATE - feat_dist_rechazo['_ult_rechazo_fecha']).dt.days
feat_dist_rechazo = feat_dist_rechazo[['feat_dist_dias_ult_rechazo']].reset_index()

# Monto máximo de los rechazos UC1: cuánto necesita el usuario para resolver su situación
df_rechazos_uc1_all = df_tx[
    (df_tx['estatus'] == 'no_procesada') &
    (df_tx['motivo_no_procesada'].isin(MOTIVOS_UC1))
]
feat_monto_max = df_rechazos_uc1_all.groupby('user_id')['monto'].max().rename('feat_monto_max_rechazo_uc1').reset_index()

print(f"feat_reintento       : {feat_reintento.shape}")
print(f"feat_dist_rechazo    : {feat_dist_rechazo.shape}")
print(f"feat_monto_max       : {feat_monto_max.shape}")

feat_reintento       : (15025, 2)
feat_dist_rechazo    : (10721, 2)
feat_monto_max       : (4733, 2)


In [8]:
# --- Construir tabla base de features por usuario ---
# Empezamos desde df_clientes para garantizar un row por cada usuario del universo
df_features_user = df_clientes[['user_id']].copy()

# Merge secuencial de todos los bloques de features de rechazo
for feat_df in [feat_rechazos_30d, feat_rechazos_90d, feat_reintento, feat_dist_rechazo, feat_monto_max]:
    df_features_user = df_features_user.merge(feat_df, on='user_id', how='left')

# Completar con 0 para usuarios sin rechazos (son la mayoría)
for col in ['feat_n_rechazos_30d', 'feat_n_rechazos_uc1_30d',
            'feat_n_rechazos_90d', 'feat_n_rechazos_uc1_90d',
            'feat_pct_reintento', 'feat_monto_max_rechazo_uc1']:
    df_features_user[col] = df_features_user[col].fillna(0)

# Cronicidad: si el usuario tiene 3+ rechazos UC1 en 90d, es un patrón establecido
df_features_user['feat_es_cronico_uc1'] = df_features_user['feat_n_rechazos_uc1_90d'] >= 3

print(f"df_features_user shape: {df_features_user.shape}")
print(f"feat_es_cronico_uc1: {df_features_user['feat_es_cronico_uc1'].sum()} usuarios crónicos")
df_features_user.head(3)

df_features_user shape: (15025, 9)
feat_es_cronico_uc1: 8 usuarios crónicos


,user_id,feat_n_rechazos_30d,feat_n_rechazos_uc1_30d,feat_n_rechazos_90d,feat_n_rechazos_uc1_90d,feat_pct_reintento,feat_dist_dias_ult_rechazo,feat_monto_max_rechazo_uc1,feat_es_cronico_uc1
0,USR-00001,0.0,0.0,2.0,1.0,0.038462,68.0,349.00,False
1,USR-00002,0.0,0.0,0.0,0.0,0.014706,125.0,339.91,False
2,USR-00003,1.0,1.0,2.0,1.0,0.014085,8.0,14750.00,False


---
## 3. Features de atipicidad (nivel transacción → nivel usuario)

Señales de comportamiento inusual que pueden indicar fraude, uso comprometido de cuenta, o patrones de gasto anómalos.

**Nivel transacción:** flag binario por fila  
**Nivel usuario:** agregaciones (porcentajes, z-scores) para el perfil de riesgo

In [9]:
# --- Features a nivel transacción ---

# Transacciones nocturnas (horas de madrugada): mayor riesgo de fraude
df_tx['feat_es_nocturna'] = df_tx['fecha_hora'].dt.hour.isin([22, 23, 0, 1, 2, 3, 4, 5])

# Transacciones internacionales: usar columna si existe, sino False
if 'es_internacional' in df_tx.columns:
    df_tx['feat_es_internacional'] = df_tx['es_internacional'].astype(bool)
else:
    # Fallback: podría detectarse por patrones en ciudad_transaccion, pero sin columna explícita = False
    df_tx['feat_es_internacional'] = False

# Combinación de internacional + nocturna: señal de mayor riesgo compuesto
df_tx['feat_intl_x_nocturna'] = df_tx['feat_es_internacional'] & df_tx['feat_es_nocturna']

# Mismatch entre ciudad de transacción y ciudad del usuario: puede indicar viaje o fraude
# Hacemos un join liviano para traer la ciudad del cliente
ciudad_user = df_clientes[['user_id', 'ciudad']].rename(columns={'ciudad': 'ciudad_cliente'})
df_tx = df_tx.merge(ciudad_user, on='user_id', how='left')
if 'ciudad_transaccion' in df_tx.columns:
    df_tx['feat_mismatch_ciudad'] = df_tx['ciudad_transaccion'] != df_tx['ciudad_cliente']
else:
    df_tx['feat_mismatch_ciudad'] = False

# Log del monto: reduce el efecto de outliers extremos en modelos lineales
df_tx['feat_log_monto'] = np.log1p(df_tx['monto'])

# Z-score del monto por (usuario, categoría MCC): detecta transacciones inusualmente grandes
# para ese usuario en esa categoría específica (vs. comparar contra toda la población)
grp_stats = df_tx.groupby(['user_id', 'categoria_mcc'])['monto'].transform
df_tx['feat_z_monto_mcc_user'] = (
    (df_tx['monto'] - grp_stats('mean')) / grp_stats('std')
).fillna(0)  # Si hay solo 1 tx en el grupo, std=NaN → ponemos 0 (no hay z-score válido)

print("Features de atipicidad a nivel transacción:")
print(f"  feat_es_nocturna     : {df_tx['feat_es_nocturna'].mean():.1%}")
print(f"  feat_es_internacional: {df_tx['feat_es_internacional'].mean():.1%}")
print(f"  feat_intl_x_nocturna : {df_tx['feat_intl_x_nocturna'].mean():.1%}")
print(f"  feat_mismatch_ciudad : {df_tx['feat_mismatch_ciudad'].mean():.1%}")

Features de atipicidad a nivel transacción:
  feat_es_nocturna     : 31.8%
  feat_es_internacional: 5.0%
  feat_intl_x_nocturna : 1.6%
  feat_mismatch_ciudad : 14.9%


In [10]:
# --- Agregaciones de atipicidad a nivel usuario ---

agg_atipicidad = df_tx.groupby('user_id').agg(
    # Porcentaje de transacciones marcadas como atípicas (señal de la plataforma)
    feat_pct_atipicas_txn       = ('patron_uso_atipico_txn', 'mean'),
    # Comportamientos derivados que calculamos nosotros
    feat_pct_nocturnas          = ('feat_es_nocturna', 'mean'),
    feat_pct_internacional      = ('feat_es_internacional', 'mean'),
    feat_pct_intl_nocturna      = ('feat_intl_x_nocturna', 'mean'),
    feat_pct_mismatch_ciudad    = ('feat_mismatch_ciudad', 'mean'),
).reset_index()

# Transacciones en disputa: señal de fraude reportado o problema con comercio
feat_disputas = df_tx[df_tx['estatus'] == 'en_disputa'].groupby('user_id').size() \
    .rename('feat_n_disputas').reset_index()

# Merge de atipicidad al df_features_user
df_features_user = df_features_user.merge(agg_atipicidad, on='user_id', how='left')
df_features_user = df_features_user.merge(feat_disputas, on='user_id', how='left')
df_features_user['feat_n_disputas'] = df_features_user['feat_n_disputas'].fillna(0).astype(int)

print(f"df_features_user shape: {df_features_user.shape}")

df_features_user shape: (15025, 15)


---
## 4. Features de liquidez

Cuantificamos la capacidad del usuario para absorber sus compromisos. Un usuario con saldo en inversión puede transferirlo para cubrir un rechazo — Havi puede actuar sobre esto.

In [11]:
# --- Features desde productos ---

# Filtrar a inversiones activas: son el colchón de liquidez disponible para el usuario
df_inv = df_productos[
    (df_productos['tipo_producto'] == 'inversion_hey') &
    (df_productos['estatus'] == 'activo')
]

feat_inv = df_inv.groupby('user_id').agg(
    # ¿Tiene al menos una inversión activa? Booleano para el modelo
    feat_tiene_inversion_activa = ('saldo_actual', lambda x: True),
    # Saldo total disponible en inversiones
    feat_saldo_inversion        = ('saldo_actual', 'sum'),
).reset_index()

# Total de productos activos: proxy de engagement con el banco
feat_n_prod = df_productos[df_productos['estatus'] == 'activo'] \
    .groupby('user_id').size().rename('feat_n_productos_activos').reset_index()

# Merge a df_features_user
df_features_user = df_features_user.merge(feat_inv, on='user_id', how='left')
df_features_user = df_features_user.merge(feat_n_prod, on='user_id', how='left')

# Usuarios sin inversión activa → False y 0
df_features_user['feat_tiene_inversion_activa'] = df_features_user['feat_tiene_inversion_activa'].fillna(False)
df_features_user['feat_saldo_inversion']         = df_features_user['feat_saldo_inversion'].fillna(0)
df_features_user['feat_n_productos_activos']     = df_features_user['feat_n_productos_activos'].fillna(0).astype(int)

# Log del saldo para reducir skew (hay usuarios con inversiones muy grandes)
df_features_user['feat_saldo_inversion_log'] = np.log1p(df_features_user['feat_saldo_inversion'])

print(f"Usuarios con inversión activa: {df_features_user['feat_tiene_inversion_activa'].sum():,}")
print(f"df_features_user shape: {df_features_user.shape}")

Usuarios con inversión activa: 4,035
df_features_user shape: (15025, 19)


In [12]:
# --- Features de carga fija desde transacciones ---

# Cargos recurrentes en últimos 30d: compromisos fijos que el usuario debe cubrir cada mes
feat_carga_fija = df_tx[
    (df_tx['fecha_hora'] >= start_30d) &
    (df_tx['tipo_operacion'] == 'cargo_recurrente')
].groupby('user_id')['monto'].sum().rename('feat_carga_fija_mensual').reset_index()

df_features_user = df_features_user.merge(feat_carga_fija, on='user_id', how='left')
df_features_user['feat_carga_fija_mensual'] = df_features_user['feat_carga_fija_mensual'].fillna(0)

# Ratio carga/ingreso: qué fracción del ingreso declarado va a compromisos fijos
# Se traen datos de ingreso desde clientes
ingreso = df_clientes[['user_id', 'ingreso_mensual_mxn']]
df_features_user = df_features_user.merge(ingreso, on='user_id', how='left')

# Clip a 5.0 para evitar que valores extremos dominen el modelo
df_features_user['feat_ratio_carga_ingreso'] = (
    df_features_user['feat_carga_fija_mensual'] / df_features_user['ingreso_mensual_mxn']
).fillna(0).clip(upper=5.0)

# Podría Havi recomendar transferir inversión para cubrir el rechazo pendiente?
# Solo aplica si hay un monto de rechazo UC1 pendiente
df_features_user['feat_cubre_rechazo_pendiente'] = (
    (df_features_user['feat_monto_max_rechazo_uc1'] > 0) &
    (df_features_user['feat_saldo_inversion'] >= df_features_user['feat_monto_max_rechazo_uc1'])
)

print(f"Usuarios que pueden cubrir rechazo con inversión: {df_features_user['feat_cubre_rechazo_pendiente'].sum():,}")
print(f"df_features_user shape: {df_features_user.shape}")

Usuarios que pueden cubrir rechazo con inversión: 1,113
df_features_user shape: (15025, 23)


---
## 5. Features de conversación (señales UC4 → UC1)

Las conversaciones del usuario con Havi contienen señales explícitas de problemas financieros. Un usuario que menciona 'rechazo' o 'fraude' en el chat ya está en modo de alerta — UC1 puede proactivizarse aún más.

Estas features conectan el dataset conversacional (UC4) con el perfil de riesgo (UC1).

In [13]:
%%time
# --- Carga del parquet de conversaciones ---
df_conv = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")

# Eliminar 18 duplicados conocidos antes de cualquier operación
n_antes_conv = len(df_conv)
df_conv = df_conv.drop_duplicates()
print(f"Conversaciones antes: {n_antes_conv:,} → después de drop_duplicates: {len(df_conv):,}  (eliminados: {n_antes_conv - len(df_conv)})")

# channel_source llega como string '1'/'2' — mantenerlo así por consistencia con UC4
df_conv['channel_source'] = df_conv['channel_source'].astype(str)

# Filtrar al CUTOFF_DATE si hay columna de fecha
if 'fecha_hora' in df_conv.columns:
    df_conv['fecha_hora'] = pd.to_datetime(df_conv['fecha_hora'])
    df_conv = df_conv[df_conv['fecha_hora'] <= CUTOFF_DATE]
    print(f"Conversaciones filtradas al CUTOFF_DATE: {len(df_conv):,}")

print(f"\nColumnas: {df_conv.columns.tolist()}")
df_conv.head(2)

Conversaciones antes: 49,999 → después de drop_duplicates: 49,981  (eliminados: 18)

Columnas: ['input', 'output', 'date', 'conv_id', 'user_id', 'channel_source']
CPU times: user 56.7 ms, sys: 17.2 ms, total: 73.9 ms
Wall time: 73.4 ms


,input,output,date,conv_id,user_id,channel_source
0,"Me enteré de una promo ""Supercashback Pagos Gu...",Claro que puedo ayudarte! Para participar en l...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1
1,La tarjeta de crédito Hey Negocios es diferent...,Claro! La Tarjeta de Crédito Hey Negocios es d...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1


In [14]:
# --- Extracción de señales UC1 desde conversaciones ---

# Identificamos la columna de texto del mensaje del usuario
# En el parquet viene como 'input' o 'user_input' según la versión
text_col = 'input' if 'input' in df_conv.columns else 'user_input' if 'user_input' in df_conv.columns else None
print(f"Columna de texto detectada: {text_col}")

if text_col:
    # Normalizar texto a minúsculas para búsqueda case-insensitive
    df_conv['_texto_lower'] = df_conv[text_col].fillna('').str.lower()

    # Keywords que indican problemas de rechazo/fraude en el contexto de UC1
    KW_UC1       = ['rechazo', 'no me pasó', 'no se procesó', 'cargo no reconocido', 'fraude', 'bloqueo', 'disputa']
    KW_FRAUDE    = ['fraude', 'no reconozco', 'desconozco']
    KW_RECHAZO   = ['rechazo', 'no procesada', 'no pasó']

    def contains_any(series, keywords):
        """True si el texto contiene al menos uno de los keywords."""
        pattern = '|'.join(keywords)
        return series.str.contains(pattern, na=False)

    # Flag por turno de conversación
    df_conv['_es_uc1']      = contains_any(df_conv['_texto_lower'], KW_UC1)
    df_conv['_es_fraude']   = contains_any(df_conv['_texto_lower'], KW_FRAUDE)
    df_conv['_es_rechazo']  = contains_any(df_conv['_texto_lower'], KW_RECHAZO)

    # Agregación por usuario
    feat_conv = df_conv.groupby('user_id').agg(
        # Cuántos mensajes del usuario mencionan problemas UC1
        feat_conv_n_mensajes_uc1      = ('_es_uc1',    'sum'),
        # ¿Mencionó fraude en algún momento?
        feat_conv_menciona_fraude     = ('_es_fraude', 'any'),
        # ¿Mencionó rechazo en algún momento?
        feat_conv_menciona_rechazo    = ('_es_rechazo', 'any'),
    ).reset_index()

    # Merge al perfil de usuario — left join para no perder usuarios sin conversaciones
    df_features_user = df_features_user.merge(feat_conv, on='user_id', how='left')
    df_features_user['feat_conv_n_mensajes_uc1']   = df_features_user['feat_conv_n_mensajes_uc1'].fillna(0).astype(int)
    df_features_user['feat_conv_menciona_fraude']  = df_features_user['feat_conv_menciona_fraude'].fillna(False)
    df_features_user['feat_conv_menciona_rechazo'] = df_features_user['feat_conv_menciona_rechazo'].fillna(False)

    print(f"Usuarios con mensajes UC1 en conversaciones : {(df_features_user['feat_conv_n_mensajes_uc1'] > 0).sum():,}")
    print(f"Usuarios que mencionaron fraude             : {df_features_user['feat_conv_menciona_fraude'].sum():,}")
    print(f"Usuarios que mencionaron rechazo            : {df_features_user['feat_conv_menciona_rechazo'].sum():,}")
else:
    # Si no se encuentra la columna de texto, agregamos las features como 0/False
    print("⚠️  Columna de texto no encontrada — features de conversación se setean en 0/False")
    df_features_user['feat_conv_n_mensajes_uc1']   = 0
    df_features_user['feat_conv_menciona_fraude']  = False
    df_features_user['feat_conv_menciona_rechazo'] = False

print(f"\ndf_features_user shape: {df_features_user.shape}")

Columna de texto detectada: input
Usuarios con mensajes UC1 en conversaciones : 371
Usuarios que mencionaron fraude             : 286
Usuarios que mencionaron rechazo            : 33

df_features_user shape: (15025, 26)


---
## 6. Tabla de alertas nivel transacción

Construimos la tabla `df_alertas` que el modelo de UC1 usará como input para generar alertas individuales. Incluye solo transacciones accionables: rechazos UC1, disputas y atipicidades.

Esta tabla es más granular que `df_features_user` — es la base para el JSON de alertas final.

In [15]:
# --- Filtro para transacciones accionables por UC1 ---

# Condición 1: rechazos que Havi puede ayudar a resolver
mask_rechazo_uc1 = (
    (df_tx['estatus'] == 'no_procesada') &
    (df_tx['motivo_no_procesada'].isin(MOTIVOS_UC1))
)

# Condición 2: transacciones en disputa (posible fraude)
mask_disputa = df_tx['estatus'] == 'en_disputa'

# Condición 3: transacciones marcadas como atípicas por la plataforma
mask_atipica = df_tx['patron_uso_atipico_txn'] == True

# Unión de todas las condiciones accionables
df_alertas = df_tx[mask_rechazo_uc1 | mask_disputa | mask_atipica].copy()

# Seleccionar columnas relevantes para la tabla de alertas
cols_alertas = [
    'transaccion_id', 'user_id', 'fecha_hora', 'monto',
    'feat_log_monto', 'feat_es_nocturna', 'feat_es_internacional',
    'feat_intl_x_nocturna', 'feat_z_monto_mcc_user',
    'estatus', 'motivo_no_procesada', 'patron_uso_atipico_txn',
    'tipo_operacion', 'categoria_mcc'
]

# Solo incluir columnas que existen en el dataframe
cols_alertas = [c for c in cols_alertas if c in df_alertas.columns]
df_alertas = df_alertas[cols_alertas].reset_index(drop=True)

print(f"df_alertas shape: {df_alertas.shape}")
print(f"\nDistribución por tipo de alerta:")
print(f"  Rechazos UC1 : {mask_rechazo_uc1.sum():,}")
print(f"  En disputa   : {mask_disputa.sum():,}")
print(f"  Atípicas     : {mask_atipica.sum():,}")
df_alertas.head(3)

df_alertas shape: (60739, 14)

Distribución por tipo de alerta:
  Rechazos UC1 : 6,059
  En disputa   : 17,300
  Atípicas     : 38,601


,transaccion_id,user_id,fecha_hora,monto,feat_log_monto,feat_es_nocturna,feat_es_internacional,feat_intl_x_nocturna,feat_z_monto_mcc_user,estatus,motivo_no_procesada,patron_uso_atipico_txn,tipo_operacion,categoria_mcc
0,TXN-0000000050,USR-00001,2025-08-23 20:37:31,349.00,5.857933,False,False,False,-0.315578,no_procesada,saldo_insuficiente,False,cargo_recurrente,servicios_digitales
1,TXN-0000000072,USR-00002,2025-05-21 00:25:13,695.07,6.545450,True,False,False,0.769775,en_disputa,NaN,False,compra,supermercado
2,TXN-0000000061,USR-00002,2025-06-27 15:37:01,339.91,5.831619,False,False,False,-0.638910,no_procesada,saldo_insuficiente,False,compra,restaurante


---
## 6.5 Features de saldo alternativo y monto faltante

Features centrales para el trigger de Havi en UC1:

- **`tiene_saldo_alternativo`**: dado un rechazo por `saldo_insuficiente`, ¿puede el usuario cubrirlo desde otro producto activo?
- **`producto_alternativo_id`**: el producto con mayor saldo disponible para cubrir el rechazo.
- **`monto_disponible_alternativo`**: saldo de ese producto alternativo.
- **`monto_faltante`**: exactamente cuánto le falta al usuario para completar la compra.

Estos features se calculan a nivel de transacción rechazada y luego se agregan al perfil de usuario (`df_features_user`).

In [ ]:
%%time
# --- Feature: tiene_saldo_alternativo ---
# Para cada rechazo por saldo_insuficiente: buscar si hay otro producto activo del mismo
# usuario cuyo saldo_actual >= monto rechazado.

rechazos_si = df_tx[
    (df_tx['estatus'] == 'no_procesada') &
    (df_tx['motivo_no_procesada'] == 'saldo_insuficiente')
][['transaccion_id', 'user_id', 'monto', 'producto_id']].copy()

prods_activos = df_productos[df_productos['estatus'] == 'activo'][
    ['user_id', 'producto_id', 'tipo_producto', 'saldo_actual']
].copy()

# Producto_id de rechazo × todos los productos activos del mismo usuario
merged_alt = rechazos_si.merge(prods_activos, on='user_id', suffixes=('_origen', '_alt'))

# Excluir el producto que ya falló (no puede cubrirse a sí mismo)
merged_alt = merged_alt[merged_alt['producto_id_origen'] != merged_alt['producto_id_alt']]

# ¿El producto alternativo cubre el monto?
merged_alt['_cubre'] = merged_alt['saldo_actual'] >= merged_alt['monto']

# Por cada rechazo: quedarse con el producto alternativo de mayor saldo
idx_best = merged_alt.groupby('transaccion_id')['saldo_actual'].idxmax()
feat_saldo_alt_txn = merged_alt.loc[idx_best, [
    'transaccion_id', 'producto_id_alt', 'tipo_producto', 'saldo_actual', '_cubre'
]].rename(columns={
    'producto_id_alt': 'feat_producto_alternativo_id',
    'tipo_producto'  : 'feat_tipo_producto_alternativo',
    'saldo_actual'   : 'feat_monto_disponible_alternativo',
    '_cubre'         : 'feat_tiene_saldo_alternativo',
}).reset_index(drop=True)

# Cobertura: % de rechazos resolubles automáticamente
n_rechazos_si  = len(rechazos_si)
n_resolubles   = int(feat_saldo_alt_txn['feat_tiene_saldo_alternativo'].sum())
n_sin_alt      = n_rechazos_si - len(feat_saldo_alt_txn)   # rechazos sin ningún producto alternativo
pct_cobertura  = n_resolubles / n_rechazos_si if n_rechazos_si > 0 else 0

print(f"Rechazos por saldo_insuficiente       : {n_rechazos_si:,}")
print(f"Sin ningún producto alternativo       : {n_sin_alt:,}")
print(f"Con alternativo que cubre el monto    : {n_resolubles:,}  ({pct_cobertura:.1%})")
print(f"Con alternativo pero saldo insuficiente: {len(feat_saldo_alt_txn) - n_resolubles:,}")
feat_saldo_alt_txn.head(3)

In [ ]:
# --- Feature: monto_faltante ---
# ¿Cuánto le falta exactamente al usuario para completar la compra rechazada?
# Se calcula como: monto_transaccion - saldo_actual_del_producto_origen (clipeado a 0).

saldo_origen = df_productos[['producto_id', 'saldo_actual']].rename(
    columns={'saldo_actual': 'saldo_origen'}
)

rechazos_con_prod = rechazos_si.merge(saldo_origen, on='producto_id', how='left')

# monto_faltante = lo que le falta; clip(lower=0) por si el saldo del snapshot ya cambió
rechazos_con_prod['feat_monto_faltante'] = (
    rechazos_con_prod['monto'] - rechazos_con_prod['saldo_origen']
).clip(lower=0)

# Normalizar por ingreso mensual para poder comparar entre perfiles de usuario
rechazos_con_prod = rechazos_con_prod.merge(
    df_clientes[['user_id', 'ingreso_mensual_mxn']], on='user_id', how='left'
)
rechazos_con_prod['feat_monto_faltante_pct_ingreso'] = (
    rechazos_con_prod['feat_monto_faltante'] / rechazos_con_prod['ingreso_mensual_mxn']
).fillna(0).clip(upper=5.0)

# Flag: resolución parcial — hay saldo alternativo pero no cubre el total
rechazos_con_prod = rechazos_con_prod.merge(
    feat_saldo_alt_txn[['transaccion_id', 'feat_tiene_saldo_alternativo', 'feat_monto_disponible_alternativo']],
    on='transaccion_id', how='left'
)
rechazos_con_prod['feat_es_resolucion_parcial'] = (
    rechazos_con_prod['feat_monto_disponible_alternativo'].gt(0) &
    rechazos_con_prod['feat_tiene_saldo_alternativo'].eq(False)
).fillna(False)

# Tabla de features por transacción (para enriquecer df_alertas)
feat_monto_faltante_txn = rechazos_con_prod[[
    'transaccion_id', 'feat_monto_faltante',
    'feat_monto_faltante_pct_ingreso', 'feat_es_resolucion_parcial'
]]

# Distribución del monto faltante
print("Distribución de feat_monto_faltante (MXN):")
print(rechazos_con_prod['feat_monto_faltante'].describe(percentiles=[0.25, 0.5, 0.75, 0.90]).to_string())
print(f"\n  < $500  MXN  : {(rechazos_con_prod['feat_monto_faltante'] < 500).mean():.1%}")
print(f"  $500–$2000   : {((rechazos_con_prod['feat_monto_faltante'] >= 500) & (rechazos_con_prod['feat_monto_faltante'] <= 2000)).mean():.1%}")
print(f"  > $2000 MXN  : {(rechazos_con_prod['feat_monto_faltante'] > 2000).mean():.1%}")
print(f"\n  Resolución parcial (alt. existe pero insuficiente): {rechazos_con_prod['feat_es_resolucion_parcial'].sum():,}")

In [ ]:
# --- Enriquecer df_alertas con features de saldo alternativo y monto faltante ---
# Solo aplican a rechazos por saldo_insuficiente; las demás filas quedan NaN → correcto.

df_alertas = df_alertas.merge(feat_saldo_alt_txn, on='transaccion_id', how='left')
df_alertas = df_alertas.merge(feat_monto_faltante_txn, on='transaccion_id', how='left')

df_alertas['feat_tiene_saldo_alternativo']   = df_alertas['feat_tiene_saldo_alternativo'].fillna(False)
df_alertas['feat_monto_faltante']            = df_alertas['feat_monto_faltante'].fillna(0.0)
df_alertas['feat_es_resolucion_parcial']     = df_alertas['feat_es_resolucion_parcial'].fillna(False)

# --- Agregar a nivel usuario (para df_features_user) ---

# Traer user_id a feat_saldo_alt_txn para poder agrupar
feat_saldo_alt_user = feat_saldo_alt_txn.merge(
    rechazos_si[['transaccion_id', 'user_id']], on='transaccion_id', how='left'
).groupby('user_id').agg(
    # % de rechazos del usuario que son resolubles automáticamente
    feat_pct_rechazos_resolubles    = ('feat_tiene_saldo_alternativo', 'mean'),
    # Máximo saldo disponible en algún producto alternativo
    feat_max_monto_disponible_alt   = ('feat_monto_disponible_alternativo', 'max'),
).reset_index()

feat_monto_faltante_user = rechazos_con_prod.groupby('user_id').agg(
    feat_monto_faltante_mediano     = ('feat_monto_faltante', 'median'),
    feat_monto_faltante_max         = ('feat_monto_faltante', 'max'),
    feat_tiene_resolucion_parcial   = ('feat_es_resolucion_parcial', 'any'),
).reset_index()

# Merge al perfil de usuario
df_features_user = df_features_user.merge(feat_saldo_alt_user,    on='user_id', how='left')
df_features_user = df_features_user.merge(feat_monto_faltante_user, on='user_id', how='left')

for col in ['feat_pct_rechazos_resolubles', 'feat_max_monto_disponible_alt',
            'feat_monto_faltante_mediano', 'feat_monto_faltante_max']:
    df_features_user[col] = df_features_user[col].fillna(0.0)
df_features_user['feat_tiene_resolucion_parcial'] = (
    df_features_user['feat_tiene_resolucion_parcial'].fillna(False)
)

print(f"df_alertas enriched shape   : {df_alertas.shape}")
print(f"df_features_user shape      : {df_features_user.shape}")
print(f"\nUsuarios con rechazos resolubles   : {(df_features_user['feat_pct_rechazos_resolubles'] > 0).sum():,}")
print(f"Usuarios con resolución parcial    : {df_features_user['feat_tiene_resolucion_parcial'].sum():,}")
print(f"Usuarios con saldo alt > 0         : {(df_features_user['feat_max_monto_disponible_alt'] > 0).sum():,}")

---
## 7. Validación post-FE

Verificaciones de sanidad antes de persistir. Mejor detectar problemas acá que en el modelo.

In [16]:
print("=" * 60)
print("VALIDACIÓN POST FEATURE ENGINEERING")
print("=" * 60)

# --- Shape esperada ---
EXPECTED_ROWS = 15_025
actual_rows = len(df_features_user)
status = "✅" if actual_rows == EXPECTED_ROWS else "⚠️ "
print(f"\n{status} Shape df_features_user: {df_features_user.shape}  (esperado: {EXPECTED_ROWS} filas)")

# --- Tasa de nulos por feature ---
feat_cols = [c for c in df_features_user.columns if c.startswith('feat_')]
null_rates = df_features_user[feat_cols].isnull().mean().sort_values(ascending=False)

print("\n--- Top 10 features por tasa de nulos ---")
print(null_rates.head(10).to_string())

high_null = null_rates[null_rates > 0.30]
if len(high_null) > 0:
    print(f"\n⚠️  {len(high_null)} features con >30% nulos — revisar:")
    print(high_null.to_string())
else:
    print("\n✅ Ninguna feature supera el 30% de nulos")

VALIDACIÓN POST FEATURE ENGINEERING

✅ Shape df_features_user: (15025, 26)  (esperado: 15025 filas)

--- Top 10 features por tasa de nulos ---
feat_dist_dias_ult_rechazo      0.286456
feat_n_rechazos_30d             0.000000
feat_n_disputas                 0.000000
feat_conv_menciona_fraude       0.000000
feat_conv_n_mensajes_uc1        0.000000
feat_cubre_rechazo_pendiente    0.000000
feat_ratio_carga_ingreso        0.000000
feat_carga_fija_mensual         0.000000
feat_saldo_inversion_log        0.000000
feat_n_productos_activos        0.000000

✅ Ninguna feature supera el 30% de nulos


In [17]:
# --- Sanity check lógico: 30d ≤ 90d ---
# Un usuario no puede tener más rechazos en 30d que en 90d
check = (df_features_user['feat_n_rechazos_30d'] <= df_features_user['feat_n_rechazos_90d']).all()
print(f"{'✅' if check else '❌'} feat_n_rechazos_30d <= feat_n_rechazos_90d para todos los usuarios: {check}")

check_uc1 = (df_features_user['feat_n_rechazos_uc1_30d'] <= df_features_user['feat_n_rechazos_uc1_90d']).all()
print(f"{'✅' if check_uc1 else '❌'} feat_n_rechazos_uc1_30d <= feat_n_rechazos_uc1_90d para todos los usuarios: {check_uc1}")

# --- Distribución de cronicidad ---
n_cronicos = df_features_user['feat_es_cronico_uc1'].sum()
pct_cronicos = n_cronicos / len(df_features_user)
print(f"\n✅ feat_es_cronico_uc1: {n_cronicos:,} usuarios ({pct_cronicos:.1%} del total)")

# --- Listado completo de features generadas ---
print(f"\n--- Features generadas ({len(feat_cols)} total) ---")
for f in sorted(feat_cols):
    dtype = df_features_user[f].dtype
    nulls = df_features_user[f].isnull().sum()
    print(f"  {f:<45} dtype={str(dtype):<10} nulls={nulls}")

✅ feat_n_rechazos_30d <= feat_n_rechazos_90d para todos los usuarios: True
✅ feat_n_rechazos_uc1_30d <= feat_n_rechazos_uc1_90d para todos los usuarios: True

✅ feat_es_cronico_uc1: 8 usuarios (0.1% del total)

--- Features generadas (24 total) ---
  feat_carga_fija_mensual                       dtype=float64    nulls=0
  feat_conv_menciona_fraude                     dtype=bool       nulls=0
  feat_conv_menciona_rechazo                    dtype=bool       nulls=0
  feat_conv_n_mensajes_uc1                      dtype=int64      nulls=0
  feat_cubre_rechazo_pendiente                  dtype=bool       nulls=0
  feat_dist_dias_ult_rechazo                    dtype=float64    nulls=4304
  feat_es_cronico_uc1                           dtype=bool       nulls=0
  feat_monto_max_rechazo_uc1                    dtype=float64    nulls=0
  feat_n_disputas                               dtype=int64      nulls=0
  feat_n_productos_activos                      dtype=int64      nulls=0
  feat_n_rechazos_

---
## 8. Persistencia

Guardamos los dos artefactos de salida que consumen los notebooks de modelado:
- `feat_uc1_user_risk.parquet`: perfil de riesgo por usuario (input del modelo de scoring)
- `feat_uc1_alertas.parquet`: transacciones accionables (input para generación de alertas)

In [18]:
# Persistir los artefactos de features para el pipeline de modelado
df_features_user.to_parquet(OUTPUT_DIR / "feat_uc1_user_risk.parquet", index=False)
df_alertas.to_parquet(OUTPUT_DIR / "feat_uc1_alertas.parquet", index=False)

print(f"✅ feat_uc1_user_risk.parquet : {df_features_user.shape}")
print(f"✅ feat_uc1_alertas.parquet   : {df_alertas.shape}")
print(f"\nOutputs en: {OUTPUT_DIR}")

✅ feat_uc1_user_risk.parquet : (15025, 26)
✅ feat_uc1_alertas.parquet   : (60739, 14)

Outputs en: /Users/diegodq/Documents/dev/datamoles/Datathon-2026/outputs/features
